# Series de Tiempo - Vías de Ingreso

En este laboratorio se construyen y analizan las series mensuales para cada vía de ingreso:
1. **Aérea**
2. **Terrestre**
3. **Marítima**

El proceso incluye:
- Instalación e importación de librerías (`pmdarima` y `prophet`).
- Interpretación detallada de gráficos (tendencia, estacionalidad, pandemia).
- Justificación de transformaciones logarítmicas.
- Análisis visual (ACF, PACF) y automático (`auto_arima`) para parámetros ARIMA/SARIMA.
- Evaluación de los residuos del modelo.
- Construcción y comparación de modelos (ARIMA, Prophet, Holt-Winters, Seasonal Naïve).
- Tabla comparativa final y conclusión para cada serie.

In [19]:
!pip install pmdarima prophet tabulate

zsh:1: command not found: pip


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pmdarima as pm
from prophet import Prophet
from IPython.display import Markdown, display

print('Librerías cargadas correctamente.')

## 1. Carga y preparación de datos

In [ ]:
# Leer el CSV generado desde el Excel original
df = pd.read_csv('./Base_Migracion_2009-2026jun.csv')
print('Shape del dataset:', df.shape)
print('\nColumnas:', list(df.columns))

In [ ]:
# Construir las 3 series mensuales agrupando por Año, Mes cod y Vía
series_data = df.groupby(['Año', 'Mes cod', 'Vía'])['Viajero'].sum().reset_index()

# Crear columna de fecha
series_data['Fecha'] = pd.to_datetime(
    series_data['Año'].astype(str) + '-' + series_data['Mes cod'].astype(str).str.zfill(2) + '-01'
)

# Separar en 3 series asegurando frecuencia mensual y rellenando posibles huecos
vias = ['Aérea', 'Terrestre', 'Marítima']
series = {}
for via in vias:
    s = series_data[series_data['Vía'] == via][['Fecha', 'Viajero']].copy()
    s = s.set_index('Fecha')
    s = s.sort_index()
    s = s.asfreq('MS')  # Frecuencia Mensual Start
    s = s.ffill()  # Rellenar posibles NaN si hay meses faltantes en los datos
    s.columns = [via]
    series[via] = s
    print(f'\n=== Serie: {via} ===')
    print(f'  Inicio:      {s.index.min().strftime("%Y-%m")}')
    print(f'  Fin:         {s.index.max().strftime("%Y-%m")}')
    print(f'  Frecuencia:  Mensual (12 observaciones por año)')
    print(f'  Total obs:   {len(s)}')

### División en entrenamiento (~70%) y prueba (~30%)

In [ ]:
train_series = {}
test_series = {}

for via in vias:
    s = series[via]
    train_size = int(len(s) * 0.7)
    train_series[via] = s.iloc[:train_size]
    test_series[via] = s.iloc[train_size:]
    print(f'\n--- {via} ---')
    print(f'  Train: {len(train_series[via])} obs ({train_series[via].index.min().strftime("%Y-%m")} a {train_series[via].index.max().strftime("%Y-%m")})')
    print(f'  Test:  {len(test_series[via])} obs ({test_series[via].index.min().strftime("%Y-%m")} a {test_series[via].index.max().strftime("%Y-%m")})')

In [ ]:
def calcular_metricas(actual, predicho):
    """Calcula MAE y RMSE para arrays alineados"""
    actual_np = np.array(actual)
    pred_np = np.array(predicho)
    mask = ~np.isnan(actual_np) & ~np.isnan(pred_np)
    if not np.any(mask):
        return np.nan, np.nan
    mae = mean_absolute_error(actual_np[mask], pred_np[mask])
    rmse = np.sqrt(mean_squared_error(actual_np[mask], pred_np[mask]))
    return mae, rmse

---
---
# SERIE: VÍA AÉREA

### a) Gráfico de la serie e interpretación — Aérea

In [ ]:
ts_aérea = train_series['Aérea']['Aérea']
plt.figure(figsize=(15, 5))
plt.plot(ts_aérea)
plt.title('Serie de Tiempo - Viajeros por Vía Aérea (Train)')
plt.xlabel('Fecha')
plt.ylabel('Viajeros')
plt.grid(True, alpha=0.3)
plt.show()

### b) Descomposición — Aérea

In [ ]:
decomp_aérea = seasonal_decompose(ts_aérea, model='multiplicative', period=12)
decomp_aérea.plot()
plt.gcf().set_size_inches(12, 8)
plt.show()

### c) Transformación Logarítmica — Aérea
**Justificación:** Se aplica una transformación logarítmica debido a que la variabilidad (varianza) aumenta conforme crece el nivel de la serie, como se observó en la gráfica original. Esta transformación ayuda a estabilizar la varianza, comprimiendo los picos altos, y facilita el modelado logrando estacionariedad en varianza.

In [ ]:
# Previniendo log(0) sumando una constante pequeña si hay ceros
ts_log_aérea = np.log(ts_aérea.replace(0, 1))

plt.figure(figsize=(15, 4))
plt.plot(ts_log_aérea)
plt.title('Serie Logarítmica (Varianza Estabilizada) - ' + 'Aérea')
plt.grid(True, alpha=0.3)
plt.show()

### d) Estacionariedad en Media (ADF) — Aérea
Corroboramos estacionariedad en media usando ADF. Si p-value > 0.05 la serie es no estacionaria en media.

In [ ]:
# Test ADF sin diferenciar
dftest = adfuller(ts_log_aérea.dropna())
print('P-value ADF (Sin diferenciar):', dftest[1])

# Test ADF con 1 diferenciación
dftest_diff = adfuller(ts_log_aérea.diff().dropna())
print('P-value ADF (1 Diferenciación):', dftest_diff[1])

if dftest_diff[1] > 0.05:
    dftest_diff2 = adfuller(ts_log_aérea.diff().diff().dropna())
    print('P-value ADF (2 Diferenciaciones):', dftest_diff2[1])
    d_opt = 2
else:
    d_opt = 1
print(f"\nSe observa que la serie alcanza la estacionariedad con d={d_opt} (p-value < 0.05)")

### e) Funciones ACF y PACF — Aérea
Se analizan los gráficos de la serie diferenciada para estimar los parámetros $p$ y $q$:
- **PACF (Autocorrelación Parcial):** Ayuda a determinar el parámetro $p$ (AR). Observamos en qué rezago se corta o decae el gráfico dentro del intervalo de confianza.
- **ACF (Autocorrelación):** Ayuda a determinar el parámetro $q$ (MA). Observamos en qué rezago decae a cero significativamente.

In [ ]:
ts_estacionaria_aérea = ts_log_aérea.diff(d_opt).dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
# ACF
pd.plotting.autocorrelation_plot(ts_estacionaria_aérea, ax=axes[0])
axes[0].set_title('ACF (Autocorrelación)')
axes[0].set_xlim(0, 36)

# PACF
from statsmodels.graphics.tsaplots import plot_pacf
plot_pacf(ts_estacionaria_aérea, lags=36, ax=axes[1], method='ywm')
axes[1].set_title('PACF (Autocorrelación Parcial)')
plt.show()

### f) Selección de Parámetros con `auto_arima` — Aérea
En lugar de proponer modelos a ciegas, `auto_arima` explora el espacio de parámetros justificando la elección final estadísticamente mediante el criterio AIC.

In [ ]:
print('Buscando el mejor modelo SARIMA para Aérea...')
auto_model_aérea = pm.auto_arima(ts_log_aérea, 
                              seasonal=True, m=12,
                              d=d_opt, D=1,
                              max_p=3, max_q=3, max_P=2, max_Q=2,
                              trace=True,
                              error_action='ignore',  
                              suppress_warnings=True, 
                              stepwise=True)

print('\nResumen del mejor modelo encontrado por auto_arima:')
print(auto_model_aérea.summary())

### g) Análisis de Residuos del modelo seleccionado — Aérea

In [ ]:
auto_model_aérea.plot_diagnostics(figsize=(15, 8))
plt.show()

### h) Modelos Alternativos — Aérea
Entrenaremos Prophet de Facebook, Holt-Winters y Seasonal Naïve. Comprobaremos su predicción sobre el conjunto de test para evaluar MAE y RMSE comparado con ARIMA.

In [ ]:
test_ts_aérea = test_series['Aérea']['Aérea']
n_test_aérea = len(test_ts_aérea)

# 1. Predicción ARIMA
# (Convertimos de logaritmo a escala original)
pred_log_arima = auto_model_aérea.predict(n_periods=n_test_aérea)
pred_arima_aérea = np.exp(pred_log_arima)

# 2. Holt-Winters
hw_model = ExponentialSmoothing(ts_aérea, trend='mul', seasonal='mul', seasonal_periods=12).fit(optimized=True)
pred_hw_aérea = hw_model.forecast(n_test_aérea)

# 3. Seasonal Naive
# El pronóstico es repetir el último año de entrenamiento (últimos 12 meses)
last_year = ts_aérea.iloc[-12:].values
# Repetir el patrón las veces necesarias para cubrir el test
pred_snaive_vals = np.tile(last_year, int(np.ceil(n_test_aérea / 12.0)))[:n_test_aérea]
pred_snaive_aérea = pd.Series(pred_snaive_vals, index=test_ts_aérea.index)

# 4. Prophet
prophet_df = pd.DataFrame({'ds': ts_aérea.index, 'y': ts_aérea.values})
m_prophet = Prophet(seasonality_mode='multiplicative', yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
m_prophet.fit(prophet_df)
future = m_prophet.make_future_dataframe(periods=n_test_aérea, freq='MS')
forecast = m_prophet.predict(future)
pred_prophet_aérea = forecast.set_index('ds')['yhat'].iloc[-n_test_aérea:]

print('Modelos alternativos ajustados y predicciones generadas.')

In [ ]:
plt.figure(figsize=(16, 7))
plt.plot(ts_aérea.index, ts_aérea, label='Train', color='blue')
plt.plot(test_ts_aérea.index, test_ts_aérea, label='Test (Real)', color='black', linewidth=2)

plt.plot(pred_arima_aérea.index, pred_arima_aérea, label='ARIMA (Auto)', color='red', linestyle='--')
plt.plot(pred_prophet_aérea.index, pred_prophet_aérea, label='Prophet', color='purple', linestyle='--')
plt.plot(pred_hw_aérea.index, pred_hw_aérea, label='Holt-Winters', color='green', linestyle='--')
plt.plot(pred_snaive_aérea.index, pred_snaive_aérea, label='Seasonal Naive', color='orange', linestyle='--')

plt.title('Comparación de Modelos - Vía Aérea')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### i) Tabla Comparativa de Métricas y Selección del Mejor Modelo — Aérea

In [ ]:
metricas_aérea = []

# Calcular MAE y RMSE
m_arima = calcular_metricas(test_ts_aérea, pred_arima_aérea)
m_prophet_m = calcular_metricas(test_ts_aérea, pred_prophet_aérea)
m_hw = calcular_metricas(test_ts_aérea, pred_hw_aérea)
m_snaive = calcular_metricas(test_ts_aérea, pred_snaive_aérea)

# Extraer AIC y BIC del mejor modelo ARIMA
aic_arima = auto_model_aérea.aic()
bic_arima = auto_model_aérea.bic()

metricas_aérea.append(['ARIMA', round(m_arima[0], 2), round(m_arima[1], 2), round(aic_arima, 2), round(bic_arima, 2)])
metricas_aérea.append(['Prophet', round(m_prophet_m[0], 2), round(m_prophet_m[1], 2), '—', '—'])
metricas_aérea.append(['Holt-Winters', round(m_hw[0], 2), round(m_hw[1], 2), '—', '—'])
metricas_aérea.append(['Seasonal Naive', round(m_snaive[0], 2), round(m_snaive[1], 2), '—', '—'])

df_res_aérea = pd.DataFrame(metricas_aérea, columns=['Modelo', 'MAE', 'RMSE', 'AIC', 'BIC'])
display(Markdown(df_res_aérea.to_markdown(index=False)))

In [ ]:
mejor_modelo_idx_aérea = df_res_aérea['MAE'].astype(float).idxmin()
mejor_modelo_nombre_aérea = df_res_aérea.iloc[mejor_modelo_idx_aérea]['Modelo']

print(f"\n=> El modelo seleccionado para la vía Aérea es: {mejor_modelo_nombre_aérea}")

---
---
# SERIE: VÍA TERRESTRE

### a) Gráfico de la serie e interpretación — Terrestre

In [ ]:
ts_terrestre = train_series['Terrestre']['Terrestre']
plt.figure(figsize=(15, 5))
plt.plot(ts_terrestre)
plt.title('Serie de Tiempo - Viajeros por Vía Terrestre (Train)')
plt.xlabel('Fecha')
plt.ylabel('Viajeros')
plt.grid(True, alpha=0.3)
plt.show()

### b) Descomposición — Terrestre

In [ ]:
decomp_terrestre = seasonal_decompose(ts_terrestre, model='multiplicative', period=12)
decomp_terrestre.plot()
plt.gcf().set_size_inches(12, 8)
plt.show()

### c) Transformación Logarítmica — Terrestre
**Justificación:** Se aplica una transformación logarítmica debido a que la variabilidad (varianza) aumenta conforme crece el nivel de la serie, como se observó en la gráfica original. Esta transformación ayuda a estabilizar la varianza, comprimiendo los picos altos, y facilita el modelado logrando estacionariedad en varianza.

In [ ]:
# Previniendo log(0) sumando una constante pequeña si hay ceros
ts_log_terrestre = np.log(ts_terrestre.replace(0, 1))

plt.figure(figsize=(15, 4))
plt.plot(ts_log_terrestre)
plt.title('Serie Logarítmica (Varianza Estabilizada) - ' + 'Terrestre')
plt.grid(True, alpha=0.3)
plt.show()

### d) Estacionariedad en Media (ADF) — Terrestre
Corroboramos estacionariedad en media usando ADF. Si p-value > 0.05 la serie es no estacionaria en media.

In [ ]:
# Test ADF sin diferenciar
dftest = adfuller(ts_log_terrestre.dropna())
print('P-value ADF (Sin diferenciar):', dftest[1])

# Test ADF con 1 diferenciación
dftest_diff = adfuller(ts_log_terrestre.diff().dropna())
print('P-value ADF (1 Diferenciación):', dftest_diff[1])

if dftest_diff[1] > 0.05:
    dftest_diff2 = adfuller(ts_log_terrestre.diff().diff().dropna())
    print('P-value ADF (2 Diferenciaciones):', dftest_diff2[1])
    d_opt = 2
else:
    d_opt = 1
print(f"\nSe observa que la serie alcanza la estacionariedad con d={d_opt} (p-value < 0.05)")

### e) Funciones ACF y PACF — Terrestre
Se analizan los gráficos de la serie diferenciada para estimar los parámetros $p$ y $q$:
- **PACF (Autocorrelación Parcial):** Ayuda a determinar el parámetro $p$ (AR). Observamos en qué rezago se corta o decae el gráfico dentro del intervalo de confianza.
- **ACF (Autocorrelación):** Ayuda a determinar el parámetro $q$ (MA). Observamos en qué rezago decae a cero significativamente.

In [ ]:
ts_estacionaria_terrestre = ts_log_terrestre.diff(d_opt).dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
# ACF
pd.plotting.autocorrelation_plot(ts_estacionaria_terrestre, ax=axes[0])
axes[0].set_title('ACF (Autocorrelación)')
axes[0].set_xlim(0, 36)

# PACF
from statsmodels.graphics.tsaplots import plot_pacf
plot_pacf(ts_estacionaria_terrestre, lags=36, ax=axes[1], method='ywm')
axes[1].set_title('PACF (Autocorrelación Parcial)')
plt.show()

### f) Selección de Parámetros con `auto_arima` — Terrestre
En lugar de proponer modelos a ciegas, `auto_arima` explora el espacio de parámetros justificando la elección final estadísticamente mediante el criterio AIC.

In [ ]:
print('Buscando el mejor modelo SARIMA para Terrestre...')
auto_model_terrestre = pm.auto_arima(ts_log_terrestre, 
                              seasonal=True, m=12,
                              d=d_opt, D=1,
                              max_p=3, max_q=3, max_P=2, max_Q=2,
                              trace=True,
                              error_action='ignore',  
                              suppress_warnings=True, 
                              stepwise=True)

print('\nResumen del mejor modelo encontrado por auto_arima:')
print(auto_model_terrestre.summary())

### g) Análisis de Residuos del modelo seleccionado — Terrestre

In [ ]:
auto_model_terrestre.plot_diagnostics(figsize=(15, 8))
plt.show()

### h) Modelos Alternativos — Terrestre
Entrenaremos Prophet de Facebook, Holt-Winters y Seasonal Naïve. Comprobaremos su predicción sobre el conjunto de test para evaluar MAE y RMSE comparado con ARIMA.

In [ ]:
test_ts_terrestre = test_series['Terrestre']['Terrestre']
n_test_terrestre = len(test_ts_terrestre)

# 1. Predicción ARIMA
# (Convertimos de logaritmo a escala original)
pred_log_arima = auto_model_terrestre.predict(n_periods=n_test_terrestre)
pred_arima_terrestre = np.exp(pred_log_arima)

# 2. Holt-Winters
hw_model = ExponentialSmoothing(ts_terrestre, trend='mul', seasonal='mul', seasonal_periods=12).fit(optimized=True)
pred_hw_terrestre = hw_model.forecast(n_test_terrestre)

# 3. Seasonal Naive
# El pronóstico es repetir el último año de entrenamiento (últimos 12 meses)
last_year = ts_terrestre.iloc[-12:].values
# Repetir el patrón las veces necesarias para cubrir el test
pred_snaive_vals = np.tile(last_year, int(np.ceil(n_test_terrestre / 12.0)))[:n_test_terrestre]
pred_snaive_terrestre = pd.Series(pred_snaive_vals, index=test_ts_terrestre.index)

# 4. Prophet
prophet_df = pd.DataFrame({'ds': ts_terrestre.index, 'y': ts_terrestre.values})
m_prophet = Prophet(seasonality_mode='multiplicative', yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
m_prophet.fit(prophet_df)
future = m_prophet.make_future_dataframe(periods=n_test_terrestre, freq='MS')
forecast = m_prophet.predict(future)
pred_prophet_terrestre = forecast.set_index('ds')['yhat'].iloc[-n_test_terrestre:]

print('Modelos alternativos ajustados y predicciones generadas.')

In [ ]:
plt.figure(figsize=(16, 7))
plt.plot(ts_terrestre.index, ts_terrestre, label='Train', color='blue')
plt.plot(test_ts_terrestre.index, test_ts_terrestre, label='Test (Real)', color='black', linewidth=2)

plt.plot(pred_arima_terrestre.index, pred_arima_terrestre, label='ARIMA (Auto)', color='red', linestyle='--')
plt.plot(pred_prophet_terrestre.index, pred_prophet_terrestre, label='Prophet', color='purple', linestyle='--')
plt.plot(pred_hw_terrestre.index, pred_hw_terrestre, label='Holt-Winters', color='green', linestyle='--')
plt.plot(pred_snaive_terrestre.index, pred_snaive_terrestre, label='Seasonal Naive', color='orange', linestyle='--')

plt.title('Comparación de Modelos - Vía Terrestre')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### i) Tabla Comparativa de Métricas y Selección del Mejor Modelo — Terrestre

In [ ]:
metricas_terrestre = []

# Calcular MAE y RMSE
m_arima = calcular_metricas(test_ts_terrestre, pred_arima_terrestre)
m_prophet_m = calcular_metricas(test_ts_terrestre, pred_prophet_terrestre)
m_hw = calcular_metricas(test_ts_terrestre, pred_hw_terrestre)
m_snaive = calcular_metricas(test_ts_terrestre, pred_snaive_terrestre)

# Extraer AIC y BIC del mejor modelo ARIMA
aic_arima = auto_model_terrestre.aic()
bic_arima = auto_model_terrestre.bic()

metricas_terrestre.append(['ARIMA', round(m_arima[0], 2), round(m_arima[1], 2), round(aic_arima, 2), round(bic_arima, 2)])
metricas_terrestre.append(['Prophet', round(m_prophet_m[0], 2), round(m_prophet_m[1], 2), '—', '—'])
metricas_terrestre.append(['Holt-Winters', round(m_hw[0], 2), round(m_hw[1], 2), '—', '—'])
metricas_terrestre.append(['Seasonal Naive', round(m_snaive[0], 2), round(m_snaive[1], 2), '—', '—'])

df_res_terrestre = pd.DataFrame(metricas_terrestre, columns=['Modelo', 'MAE', 'RMSE', 'AIC', 'BIC'])
display(Markdown(df_res_terrestre.to_markdown(index=False)))

In [ ]:
mejor_modelo_idx_terrestre = df_res_terrestre['MAE'].astype(float).idxmin()
mejor_modelo_nombre_terrestre = df_res_terrestre.iloc[mejor_modelo_idx_terrestre]['Modelo']

print(f"\n=> El modelo seleccionado para la vía Terrestre es: {mejor_modelo_nombre_terrestre}")

---
---
# SERIE: VÍA MARÍTIMA

### a) Gráfico de la serie e interpretación — Marítima

In [ ]:
ts_marítima = train_series['Marítima']['Marítima']
plt.figure(figsize=(15, 5))
plt.plot(ts_marítima)
plt.title('Serie de Tiempo - Viajeros por Vía Marítima (Train)')
plt.xlabel('Fecha')
plt.ylabel('Viajeros')
plt.grid(True, alpha=0.3)
plt.show()

### b) Descomposición — Marítima

In [ ]:
decomp_marítima = seasonal_decompose(ts_marítima, model='multiplicative', period=12)
decomp_marítima.plot()
plt.gcf().set_size_inches(12, 8)
plt.show()

### c) Transformación Logarítmica — Marítima
**Justificación:** Se aplica una transformación logarítmica debido a que la variabilidad (varianza) aumenta conforme crece el nivel de la serie, como se observó en la gráfica original. Esta transformación ayuda a estabilizar la varianza, comprimiendo los picos altos, y facilita el modelado logrando estacionariedad en varianza.

In [ ]:
# Previniendo log(0) sumando una constante pequeña si hay ceros
ts_log_marítima = np.log(ts_marítima.replace(0, 1))

plt.figure(figsize=(15, 4))
plt.plot(ts_log_marítima)
plt.title('Serie Logarítmica (Varianza Estabilizada) - ' + 'Marítima')
plt.grid(True, alpha=0.3)
plt.show()

### d) Estacionariedad en Media (ADF) — Marítima
Corroboramos estacionariedad en media usando ADF. Si p-value > 0.05 la serie es no estacionaria en media.

In [ ]:
# Test ADF sin diferenciar
dftest = adfuller(ts_log_marítima.dropna())
print('P-value ADF (Sin diferenciar):', dftest[1])

# Test ADF con 1 diferenciación
dftest_diff = adfuller(ts_log_marítima.diff().dropna())
print('P-value ADF (1 Diferenciación):', dftest_diff[1])

if dftest_diff[1] > 0.05:
    dftest_diff2 = adfuller(ts_log_marítima.diff().diff().dropna())
    print('P-value ADF (2 Diferenciaciones):', dftest_diff2[1])
    d_opt = 2
else:
    d_opt = 1
print(f"\nSe observa que la serie alcanza la estacionariedad con d={d_opt} (p-value < 0.05)")

### e) Funciones ACF y PACF — Marítima
Se analizan los gráficos de la serie diferenciada para estimar los parámetros $p$ y $q$:
- **PACF (Autocorrelación Parcial):** Ayuda a determinar el parámetro $p$ (AR). Observamos en qué rezago se corta o decae el gráfico dentro del intervalo de confianza.
- **ACF (Autocorrelación):** Ayuda a determinar el parámetro $q$ (MA). Observamos en qué rezago decae a cero significativamente.

In [ ]:
ts_estacionaria_marítima = ts_log_marítima.diff(d_opt).dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
# ACF
pd.plotting.autocorrelation_plot(ts_estacionaria_marítima, ax=axes[0])
axes[0].set_title('ACF (Autocorrelación)')
axes[0].set_xlim(0, 36)

# PACF
from statsmodels.graphics.tsaplots import plot_pacf
plot_pacf(ts_estacionaria_marítima, lags=36, ax=axes[1], method='ywm')
axes[1].set_title('PACF (Autocorrelación Parcial)')
plt.show()

### f) Selección de Parámetros con `auto_arima` — Marítima
En lugar de proponer modelos a ciegas, `auto_arima` explora el espacio de parámetros justificando la elección final estadísticamente mediante el criterio AIC.

In [ ]:
print('Buscando el mejor modelo SARIMA para Marítima...')
auto_model_marítima = pm.auto_arima(ts_log_marítima, 
                              seasonal=True, m=12,
                              d=d_opt, D=1,
                              max_p=3, max_q=3, max_P=2, max_Q=2,
                              trace=True,
                              error_action='ignore',  
                              suppress_warnings=True, 
                              stepwise=True)

print('\nResumen del mejor modelo encontrado por auto_arima:')
print(auto_model_marítima.summary())

### g) Análisis de Residuos del modelo seleccionado — Marítima

In [ ]:
auto_model_marítima.plot_diagnostics(figsize=(15, 8))
plt.show()

### h) Modelos Alternativos — Marítima
Entrenaremos Prophet de Facebook, Holt-Winters y Seasonal Naïve. Comprobaremos su predicción sobre el conjunto de test para evaluar MAE y RMSE comparado con ARIMA.

In [ ]:
test_ts_marítima = test_series['Marítima']['Marítima']
n_test_marítima = len(test_ts_marítima)

# 1. Predicción ARIMA
# (Convertimos de logaritmo a escala original)
pred_log_arima = auto_model_marítima.predict(n_periods=n_test_marítima)
pred_arima_marítima = np.exp(pred_log_arima)

# 2. Holt-Winters
hw_model = ExponentialSmoothing(ts_marítima, trend='mul', seasonal='mul', seasonal_periods=12).fit(optimized=True)
pred_hw_marítima = hw_model.forecast(n_test_marítima)

# 3. Seasonal Naive
# El pronóstico es repetir el último año de entrenamiento (últimos 12 meses)
last_year = ts_marítima.iloc[-12:].values
# Repetir el patrón las veces necesarias para cubrir el test
pred_snaive_vals = np.tile(last_year, int(np.ceil(n_test_marítima / 12.0)))[:n_test_marítima]
pred_snaive_marítima = pd.Series(pred_snaive_vals, index=test_ts_marítima.index)

# 4. Prophet
prophet_df = pd.DataFrame({'ds': ts_marítima.index, 'y': ts_marítima.values})
m_prophet = Prophet(seasonality_mode='multiplicative', yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
m_prophet.fit(prophet_df)
future = m_prophet.make_future_dataframe(periods=n_test_marítima, freq='MS')
forecast = m_prophet.predict(future)
pred_prophet_marítima = forecast.set_index('ds')['yhat'].iloc[-n_test_marítima:]

print('Modelos alternativos ajustados y predicciones generadas.')

In [ ]:
plt.figure(figsize=(16, 7))
plt.plot(ts_marítima.index, ts_marítima, label='Train', color='blue')
plt.plot(test_ts_marítima.index, test_ts_marítima, label='Test (Real)', color='black', linewidth=2)

plt.plot(pred_arima_marítima.index, pred_arima_marítima, label='ARIMA (Auto)', color='red', linestyle='--')
plt.plot(pred_prophet_marítima.index, pred_prophet_marítima, label='Prophet', color='purple', linestyle='--')
plt.plot(pred_hw_marítima.index, pred_hw_marítima, label='Holt-Winters', color='green', linestyle='--')
plt.plot(pred_snaive_marítima.index, pred_snaive_marítima, label='Seasonal Naive', color='orange', linestyle='--')

plt.title('Comparación de Modelos - Vía Marítima')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### i) Tabla Comparativa de Métricas y Selección del Mejor Modelo — Marítima

In [ ]:
metricas_marítima = []

# Calcular MAE y RMSE
m_arima = calcular_metricas(test_ts_marítima, pred_arima_marítima)
m_prophet_m = calcular_metricas(test_ts_marítima, pred_prophet_marítima)
m_hw = calcular_metricas(test_ts_marítima, pred_hw_marítima)
m_snaive = calcular_metricas(test_ts_marítima, pred_snaive_marítima)

# Extraer AIC y BIC del mejor modelo ARIMA
aic_arima = auto_model_marítima.aic()
bic_arima = auto_model_marítima.bic()

metricas_marítima.append(['ARIMA', round(m_arima[0], 2), round(m_arima[1], 2), round(aic_arima, 2), round(bic_arima, 2)])
metricas_marítima.append(['Prophet', round(m_prophet_m[0], 2), round(m_prophet_m[1], 2), '—', '—'])
metricas_marítima.append(['Holt-Winters', round(m_hw[0], 2), round(m_hw[1], 2), '—', '—'])
metricas_marítima.append(['Seasonal Naive', round(m_snaive[0], 2), round(m_snaive[1], 2), '—', '—'])

df_res_marítima = pd.DataFrame(metricas_marítima, columns=['Modelo', 'MAE', 'RMSE', 'AIC', 'BIC'])
display(Markdown(df_res_marítima.to_markdown(index=False)))

In [ ]:
mejor_modelo_idx_marítima = df_res_marítima['MAE'].astype(float).idxmin()
mejor_modelo_nombre_marítima = df_res_marítima.iloc[mejor_modelo_idx_marítima]['Modelo']

print(f"\n=> El modelo seleccionado para la vía Marítima es: {mejor_modelo_nombre_marítima}")